# Preprocess FlyWire Tables

One-time FlyWire preprocessing runbook. Raw table choices come from [`paths.py`](../../../src/cex/dataset/paths.py), processing logic lives in [`raw.py`](../../../src/cex/preprocessing/raw.py), and normalized output names live in [`schema.py`](../../../src/cex/dataset/schema.py).


In [1]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path

import cex
from cex import get_dataset
import cex.dataset.paths as dpaths
import cex.dataset.schema as dsch
import cex.preprocessing as dprep
import cex.preprocessing.io_stat as io_stat
import cex.util.tables as rt

DATASET_NAME = "flywire"
REPO_ROOT = Path(cex.__file__).resolve().parents[2]
DATA_ROOT = REPO_ROOT / "data" / DATASET_NAME
WRITE_PREPROCESSED_Q = True

flywire = get_dataset(DATASET_NAME, DATA_ROOT)
fp_download = flywire.paths.fp_download
fp_preprocessed = flywire.paths.fp_preprocessed
fw_files = {name: os.path.join(fp_download, fn) for name, fn in dpaths.FLYWIRE_DOWNLOAD_FILES.items()}
fp_download, fp_preprocessed


('/Users/xiangji/Documents/GitHub/codexplorer/data/flywire/download_data',
 '/Users/xiangji/Documents/GitHub/codexplorer/data/flywire/preprocessed')

## Optional Download

Set `DOWNLOAD_URL` after placing the dataset archive in Dropbox. The archive is downloaded to `download_data` and extracted there. Leave it as `None` when the raw files are already present.


In [2]:
DOWNLOAD_URL = "https://www.dropbox.com/scl/fo/a01t6njrz0hs8b7m47nys/AHSSrFfESyg903zDTCtnOx4?rlkey=gm8k4sakl70ynxoviidw2w56h&dl=1"
DOWNLOAD_ARCHIVE_NAME = "Flywire_raw_data.zip"
DOWNLOAD_OVERWRITE_Q = False

if DOWNLOAD_URL is not None:
    dprep.download_and_extract(DOWNLOAD_URL, fp_download, DOWNLOAD_ARCHIVE_NAME, DOWNLOAD_OVERWRITE_Q)


## Inspect Downloaded Tables


In [3]:
for name, fp in fw_files.items():
    print(f"\n{name}: {fp}")
    print(rt.table_columns(fp))
    display(rt.preview_table(fp, n=2))



cell_classification: /Users/xiangji/Documents/GitHub/codexplorer/data/flywire/download_data/classification.csv
['root_id', 'flow', 'super_class', 'class', 'sub_class', 'hemilineage', 'side', 'nerve']


,root_id,flow,super_class,class,sub_class,hemilineage,side,nerve
0,720575940596125868,intrinsic,optic,optic_lobe_intrinsic,t5_neuron,NaN,right,NaN
1,720575940597856265,intrinsic,optic,optic_lobe_intrinsic,transmedullary,NaN,right,NaN



cell_data: /Users/xiangji/Documents/GitHub/codexplorer/data/flywire/download_data/consolidated_cell_types.csv
['root_id', 'primary_type', 'additional_type(s)']


,root_id,primary_type,additional_type(s)
0,720575940596125868,T5c,NaN
1,720575940597856265,Tm16,NaN



neurotransmitter_data: /Users/xiangji/Documents/GitHub/codexplorer/data/flywire/download_data/neurons.csv.gz
['root_id', 'group', 'nt_type', 'nt_type_score', 'da_avg', 'ser_avg', 'gaba_avg', 'glut_avg', 'ach_avg', 'oct_avg']


...

,root_id,type,family,subsystem,category,side
0,720575940596125868,T5c,T5 Neuron,Motion,intrinsic,right
1,720575940597856265,Tm16,Transmedullary,Color,intrinsic,right



columns: /Users/xiangji/Documents/GitHub/codexplorer/data/flywire/download_data/column_assignment.csv.gz
['root_id', 'hemisphere', 'type', 'column_id', 'x', 'y', 'p', 'q']


,root_id,hemisphere,type,column_id,x,y,p,q
0,720575940596125868,right,T5c,97,-5,2,6,-4
1,720575940599333574,right,Tm1,355,-7,-6,4,-10



synapses: /Users/xiangji/Documents/GitHub/codexplorer/data/flywire/download_data/fafb_v783_princeton_synapse_table.csv.gz
['pre_x', 'pre_y', 'pre_z', 'ctr_x', 'ctr_y', 'ctr_z', 'post_x', 'post_y', 'post_z', 'size', 'pre_root_id_720575940', 'post_root_id_720575940', 'neuropil']


,pre_x,pre_y,pre_z,ctr_x,ctr_y,ctr_z,post_x,post_y,post_z,size,pre_root_id_720575940,post_root_id_720575940,neuropil
0,96224,405424,165280,96112,405424,165400,96000,405392,165360,103,610757204,620797269,LA_L
1,104880,267168,191400,104960,267296,191400,104880,267312,191360,33,631658191,615542890,NaN


## Convert `cell_data.parquet`

[`build_flywire_cell_data`](../../../src/cex/preprocessing/raw.py) merges FlyWire classification, cell type, and neurotransmitter tables, then assigns normalized `id` values while preserving source `rid`.


In [4]:
cell_classification_table = rt.read_table(fw_files["cell_classification"])
cell_table = rt.read_table(fw_files["cell_data"])
neurotransmitter_table = rt.read_table(fw_files["neurotransmitter_data"])

cell_data = dprep.build_flywire_cell_data(cell_classification_table, cell_table, neurotransmitter_table)

if WRITE_PREPROCESSED_Q:
    rt.write_table(cell_data, os.path.join(fp_preprocessed, dsch.CELL_DATA_FILE))

cell_data.head()


,id,rid,type,side,flow,super_class,class,sub_class,hemilineage,nerve,group,nt_type,nt_type_score,da_avg,ser_avg,gaba_avg,glut_avg,ach_avg,oct_avg
0,0,720575940638027497,4A0,-1,intrinsic,central,CX,tangential,putative_primary,NaN,FB,GLUT,0.50,0.09,0.02,0.10,0.50,0.28,0.01
1,1,720575940639410291,4A0,1,intrinsic,central,CX,tangential,putative_primary,NaN,FB,GLUT,0.45,0.10,0.02,0.10,0.45,0.33,0.00
2,2,720575940618308987,4A1,1,intrinsic,central,CX,tangential,putative_primary,NaN,FB,GLUT,0.77,0.04,0.03,0.02,0.77,0.14,0.00
3,3,720575940622462173,4A1,-1,intrinsic,central,CX,tangential,putative_primary,NaN,FB,GLUT,0.74,0.05,0.03,0.03,0.74,0.16,0.00
4,4,720575940608140380,4A10,1,intrinsic,central,CX,tangential,DM6__prim,NaN,FB,DA,0.63,0.63,0.27,0.01,0.02,0.06,0.01


## Convert `type_data.parquet`

[`build_type_data`](../../../src/cex/preprocessing/raw.py) summarizes type-level attributes by majority value and applies FlyWire type neurotransmitter ground truth from [`paths.py`](../../../src/cex/dataset/paths.py).


In [5]:
type_data = dprep.build_type_data(cell_data, dpaths.FLYWIRE_TYPE_COLUMNS, 
                                  col_name_map=dpaths.FLYWIRE_TYPE_COLUMN_RENAME, 
                                  nt_correction=dsch.TYPE_NEUROTRANSMITTER_GT)

if WRITE_PREPROCESSED_Q:
    rt.write_table(type_data, os.path.join(fp_preprocessed, dsch.TYPE_DATA_FILE))

type_data.head()

,type,num_cells,nt,flow,super_class,class,sub_class,hemilineage,nerve,group
0,4A0,2,GLUT,intrinsic,central,CX,tangential,putative_primary,NAN,FB
1,4A1,2,GLUT,intrinsic,central,CX,tangential,putative_primary,NAN,FB
2,4A10,4,DA,intrinsic,central,CX,tangential,DM6__prim,NAN,FB
3,4A2,16,GLUT,intrinsic,central,CX,tangential,DM6_dorso_medial,NAN,FB
4,4A21,4,GLUT,intrinsic,central,CX,tangential,DM6__prim,NAN,FB


## Convert `visual_type_data.parquet`


In [6]:
visual_cell_data = rt.read_table(fw_files["visual_cell_data"])
visual_type_data = dprep.build_type_data(visual_cell_data, 
                                         dpaths.FLYWIRE_VISUAL_TYPE_COLUMNS)
if "side" in visual_type_data: 
    visual_type_data["side"] = dprep.normalize_side_values(visual_type_data["side"])

if WRITE_PREPROCESSED_Q:
    rt.write_table(visual_type_data, os.path.join(fp_preprocessed, dsch.VISUAL_TYPE_DATA_FILE))

visual_type_data.head()

,type,num_cells,family,subsystem,category
0,5-HTPMPV01,2,other,NAN,boundary
1,5-HTPMPV03,2,other,NAN,boundary
2,5th-LNv,2,other,NAN,boundary
3,AN_multi_124,7,AN,NAN,boundary
4,AN_multi_125,2,AN,NAN,boundary


## Convert `columns_data.npz`


In [7]:
columns_raw = rt.read_table(fw_files["columns"])
column_data = dprep.build_column_data(
    columns_raw,
    cell_data,
    rid_col="root_id",
    kept_cols=dpaths.FLYWIRE_COLUMN_COLUMNS,
)

if WRITE_PREPROCESSED_Q:
    rt.write_table(column_data, os.path.join(fp_preprocessed, dsch.COLUMN_DATA_FILE))

column_data

{'id': array([106489, 109942,  98706, ..., 116175,  42289,  43693],
       shape=(45528,), dtype=uint32),
 'column_id': array([ 97, 355, 247, ..., 514, 634, 647], shape=(45528,), dtype=uint16),
 'x': array([-5, -7, -4, ..., -4, -3, -4], shape=(45528,), dtype=int8),
 'y': array([  2,  -6, -15, ..., -10, -15,  -7], shape=(45528,), dtype=int8),
 'p': array([ 6,  4, -4, ..., -1, -5,  0], shape=(45528,), dtype=int8),
 'q': array([ -4, -10, -11, ...,  -9, -10,  -7], shape=(45528,), dtype=int8),
 'side': array([ 1,  1,  1, ..., -1, -1, -1], shape=(45528,), dtype=int8),
 'pq_min': array([-19, -17], dtype=int16),
 'pq_max': array([18, 17], dtype=int16)}

## Convert `synapses.parquet`

[`build_flywire_synapse_data`](../../../src/cex/preprocessing/raw.py) applies the Princeton full-RID offset and maps source RIDs to normalized cell IDs.


In [8]:
synapse_source_table = rt.read_table(fw_files["synapses"], columns=list(dpaths.FLYWIRE_SYNAPSE_COLUMNS))
synapse_table = dprep.normalize_synapse_table(synapse_source_table, cell_data, 
                    pre_rid_col="pre_root_id_720575940", 
                    post_rid_col="post_root_id_720575940",
                    x_col="ctr_x", y_col="ctr_y", z_col="ctr_z",
                    rid_add=dpaths.FLYWIRE_PRINCETON_RID_ADD)

if WRITE_PREPROCESSED_Q:
    rt.write_table(synapse_table, os.path.join(fp_preprocessed, dsch.SYNAPSE_DATA_FILE))

synapse_table.head()

,pre_id,post_id,x_nm,y_nm,z_nm
0,0,0,559952,198016,98920
1,0,0,495312,176160,122520
2,0,0,544736,176688,136480
3,0,0,482272,119248,86520
4,0,0,537568,182384,118160


## Convert `cell_to_cell_syn_count.parquet`


In [9]:
connectivity = dprep.build_connectivity_edges(synapse_table)

if WRITE_PREPROCESSED_Q:
    rt.write_table(connectivity, os.path.join(fp_preprocessed, dsch.CELL_TO_CELL_SYN_COUNT_FILE))

connectivity.head()


,pre_id,post_id,num_syn
0,0,0,133
1,0,1,2
2,0,4,4
3,0,5,4
4,0,6,5


## Per-Cell IO Statistics

In [10]:
RUN_IO_STAT_Q = True
IO_STAT_NUM_WORKERS = 8

if RUN_IO_STAT_Q:
    io_stat.compute_all_io_stats(
        flywire,
        stat_type=dpaths.DEFAULT_IO_STAT_TYPE,
        min_syn_per_rid=1,
        min_frac=1e-2,
        num_workers=IO_STAT_NUM_WORKERS,
        write_Q=True,
    )

## Aggregate Type Connectivity

Aggregate the cell-level edge table once into a sparse type-by-type matrix. Entry `(i, j)` is the total number of synapses from `type_data['type'][i]` to `type_data['type'][j]`; the saved type list preserves that exact row and column order for direct lookup through `dataset.connectivity`.

In [11]:
type_connectivity_data = dprep.build_type_connectivity_data(
    connectivity, cell_data, type_data['type'].values)
type_connectivity_fp = os.path.join(fp_preprocessed, dsch.TYPE_TO_TYPE_SYN_COUNT_FILE)
if WRITE_PREPROCESSED_Q:
    rt.write_table(type_connectivity_data, type_connectivity_fp)

type_connectivity_fp, type_connectivity_data['shape'], len(type_connectivity_data['data'])

('/Users/xiangji/Documents/GitHub/codexplorer/data/flywire/preprocessed/type_to_type_syn_count.npz',
 array([8547, 8547]),
 2798101)